# NIFTY Gap Strategy — v7 (L1 Logistic Regression, Continuous Features)

**Key changes from v6:**

| v6 approach | v7 approach |
|---|---|
| 13 binary signals as features | 13 continuous features (returns, VIX levels, regime) |
| statsmodels Logit (MLE) | sklearn L1 LogisticRegressionCV (liblinear) |
| Perfect separation → non-convergence | Continuous features → no separation |
| Binary SGX UP/DOWN (r=−0.99) | Continuous SGX_ret (no collinearity) |
| 70-min simulation per run | Reuses sim_cache.csv from v6 (instant load) |

**Features:**
- `gap_pct`: NIFTY open gap (continuous)
- `prev_india_ret`: India close-to-close return (previous day)
- `us_ret`: composite US return = mean(SP500, NASDAQ, DOW)
- `europe_ret`: composite Europe return = mean(DAX, FTSE)
- `asia_ret`: composite Asia return = mean(NIKKEI, HANGSENG, SGX)
- `VIX_US_ret`: change in US VIX (risk appetite)
- `VIX_US_level`: absolute US VIX level (regime)
- `VIX_INDIA_level`: India VIX level (local risk regime)
- `log_entry_prem`: log(entry premium) — captures DTE/moneyness
- `dte`: days to weekly expiry (0–7)
- `nifty_20d_ret`: 20-day compounded NIFTY return (trend regime)
- `nifty_20d_realized_vol`: 20-day rolling std of daily returns (vol regime)
- `gap_normalized`: gap_pct / realized_vol (vol-adjusted gap size)

**Walk-forward split:** Train 2024–Jun 2025 | OOS Jul 2025–Mar 2026

| Parameter | Value |
|---|---|
| SL | −15% of entry premium |
| TP | +40% of entry premium |
| Breakeven win rate | 27.3% |
| Strike | ATM − 50 (1-OTM PUT) |
| Simulation | Loaded from v6/sim_cache.csv |
| Model | L1 LogisticRegressionCV (penalty='l1', solver='liblinear') |

In [1]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import date
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')

# ── Trade parameters ──────────────────────────────────────────────────────────
SL_PCT           = 0.15
TP_PCT           = 0.40
LOT_SIZE         = 75
STRIKE_STEP      = 50
BASE_LOTS        = 5
MAX_LOTS         = 25
DTE0_MAX_LOTS    = 10
STARTING_CAPITAL = 200_000.0

# ── Walk-forward split (same as v6) ──────────────────────────────────────────
TRAIN_END = date(2025, 6, 30)   # train on 2024 + first half 2025
OOS_START = date(2025, 7, 1)    # OOS = second half 2025 + 2026

# ── L1 Logistic Regression ────────────────────────────────────────────────────
# Cross-validation picks C from this list (smaller C = stronger regularization)
CV_C_GRID    = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
CLASS_WEIGHT = {0: 1.0, 1: 2.5}   # up-weight wins to compensate for ~24% base rate

# ── Threshold selection ───────────────────────────────────────────────────────
PROB_THRESHOLD    = None   # None = auto-select from training data
MIN_THRESH_TRADES = 15     # auto-select needs at least this many in-sample trades
EDGE_TARGET_PP    = 8      # auto-select targets base_win_rate + this many pp

BREAKEVEN = SL_PCT / (SL_PCT + TP_PCT)

print('Config loaded.')
print(f'Train: 2024 – {TRAIN_END}  |  OOS: {OOS_START}+')
print(f'CV C grid: {CV_C_GRID}  |  class_weight: {CLASS_WEIGHT}')
print(f'Breakeven win rate: {BREAKEVEN:.1%}')

Config loaded.
Train: 2024 – 2025-06-30  |  OOS: 2025-07-01+
CV C grid: [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]  |  class_weight: {0: 1.0, 1: 2.5}
Breakeven win rate: 27.3%


In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────────
GAP_TRADING     = Path.cwd().parent
SIM_CACHE_PATH  = GAP_TRADING / 'v6' / 'sim_cache.csv'
ALIGNED_CSV     = GAP_TRADING / 'v2' / 'v2_aligned_dataset.csv'

for lbl, p in [('sim_cache (v6)', SIM_CACHE_PATH), ('aligned CSV (v2)', ALIGNED_CSV)]:
    print(f'{lbl:<20}: {"OK" if p.exists() else "MISSING"}  ({p})')

# ── Load sim_cache (trade outcomes from v6) ───────────────────────────────────
sim_df = pd.read_csv(SIM_CACHE_PATH, parse_dates=['date'])
sim_df['date'] = sim_df['date'].dt.date
print(f'\nsim_cache : {len(sim_df)} rows  ({sim_df["date"].min()} → {sim_df["date"].max()})')
print(f'Columns   : {list(sim_df.columns[:6])} + {len(sim_df.columns)-6} signal cols')

# Keep only the columns we need from sim_cache
sim_core = sim_df[['date', 'win', 'exit_reason', 'entry_prem', 'exit_prem', 'dte']].copy()

# ── Load aligned dataset (continuous market features) ────────────────────────
aligned = pd.read_csv(ALIGNED_CSV, parse_dates=['india_date'])
aligned['india_date'] = aligned['india_date'].dt.date
aligned['VIX_INDIA_level'] = aligned['VIX_INDIA_level'].ffill().bfill()  # 49 missing rows
print(f'aligned   : {len(aligned)} rows  ({aligned["india_date"].min()} → {aligned["india_date"].max()})')

# ── Rolling regime features — computed on full aligned history before merging ─
aligned = aligned.sort_values('india_date').reset_index(drop=True)
aligned['nifty_20d_realized_vol'] = aligned['prev_india_ret'].rolling(20).std()
aligned['nifty_20d_ret'] = (
    np.exp(np.log1p(aligned['prev_india_ret']).rolling(20).sum()) - 1
)
print(f'Regime features added: nifty_20d_ret, nifty_20d_realized_vol')

# ── Merge on date (inner join) ────────────────────────────────────────────────
ALIGNED_COLS = ['india_date', 'gap_pct', 'prev_india_ret',
                'SP500_ret', 'NASDAQ_ret', 'DOW_ret',
                'DAX_ret', 'FTSE_ret',
                'NIKKEI_ret', 'HANGSENG_ret', 'SGX_ret',
                'VIX_US_ret', 'VIX_US_level', 'VIX_INDIA_level',
                'nifty_20d_ret', 'nifty_20d_realized_vol']

merged = sim_core.merge(
    aligned[ALIGNED_COLS],
    left_on='date', right_on='india_date', how='inner'
).drop(columns=['india_date'])

print(f'merged    : {len(merged)} rows  ({merged["date"].min()} → {merged["date"].max()})')

# ── Derive composite features ─────────────────────────────────────────────────
merged['us_ret']         = merged[['SP500_ret', 'NASDAQ_ret', 'DOW_ret']].mean(axis=1)
merged['europe_ret']     = merged[['DAX_ret', 'FTSE_ret']].mean(axis=1)
merged['asia_ret']       = merged[['NIKKEI_ret', 'HANGSENG_ret', 'SGX_ret']].mean(axis=1)
merged['log_entry_prem'] = np.log(merged['entry_prem'].clip(lower=0.1))
merged['gap_normalized'] = merged['gap_pct'] / merged['nifty_20d_realized_vol'].replace(0, np.nan)

# ── Feature set ───────────────────────────────────────────────────────────────
FEATURES = [
    'gap_pct',                # raw gap magnitude + direction
    'prev_india_ret',         # India momentum
    'us_ret',                 # composite US overnight return
    'europe_ret',             # composite Europe return
    'asia_ret',               # composite Asia return (incl. SGX as N225 proxy)
    'VIX_US_ret',             # VIX change (fear gauge)
    'VIX_US_level',           # absolute VIX level (regime)
    'VIX_INDIA_level',        # India VIX (local risk regime)
    'log_entry_prem',         # log(entry premium) — captures DTE + moneyness
    'dte',                    # days to expiry (0 = expiry day, typically 0-7)
    'nifty_20d_ret',          # 20-day compounded NIFTY return (trend regime)
    'nifty_20d_realized_vol', # 20-day realized volatility (vol regime)
    'gap_normalized',         # gap_pct / realized_vol (vol-adjusted gap size)
]

# Drop rows with NaN in any feature
before = len(merged)
merged = merged.dropna(subset=FEATURES).reset_index(drop=True)
print(f'\nAfter dropna: {len(merged)} rows (dropped {before - len(merged)} with missing features)')

# ── Train / OOS split ─────────────────────────────────────────────────────────
train_df = merged[merged['date'] <= TRAIN_END].reset_index(drop=True)
oos_df   = merged[merged['date'] >= OOS_START].reset_index(drop=True)

base_win_rate = train_df['win'].mean()

print(f'\nTrain : {train_df["date"].min()} → {train_df["date"].max()}  ({len(train_df)} days)')
print(f'OOS   : {oos_df["date"].min()} → {oos_df["date"].max()}  ({len(oos_df)} days)')
print(f'\nTraining outcomes:')
print(f'  Base win rate : {base_win_rate:.1%}  (breakeven: {BREAKEVEN:.1%})')
print(train_df['exit_reason'].value_counts().to_string())

sim_cache (v6)      : OK  (c:\Users\sayan\OneDrive\Desktop\Projects\03_Market_Research\market-research\gap_trading\v6\sim_cache.csv)
aligned CSV (v2)    : OK  (c:\Users\sayan\OneDrive\Desktop\Projects\03_Market_Research\market-research\gap_trading\v2\v2_aligned_dataset.csv)

sim_cache : 402 rows  (2024-01-02 → 2026-03-24)
Columns   : ['date', 'win', 'exit_reason', 'entry_prem', 'exit_prem', 'dte'] + 13 signal cols
aligned   : 740 rows  (2023-03-31 → 2026-04-02)
Regime features added: nifty_20d_ret, nifty_20d_realized_vol
merged    : 402 rows  (2024-01-02 → 2026-03-24)

After dropna: 382 rows (dropped 20 with missing features)

Train : 2024-01-02 → 2025-06-27  (254 days)
OOS   : 2025-07-01 → 2026-03-24  (128 days)

Training outcomes:
  Base win rate : 24.4%  (breakeven: 27.3%)
exit_reason
Stop Loss     180
Target Hit     62
11:15 exit     12


In [3]:
# ── Prepare feature matrices ───────────────────────────────────────────────────
X_train = train_df[FEATURES].values
y_train = train_df['win'].astype(int).values
X_oos   = oos_df[FEATURES].values

# Scale features — required for L1 (ensures regularization is applied uniformly)
scaler   = StandardScaler()
Xs_train = scaler.fit_transform(X_train)
Xs_oos   = scaler.transform(X_oos)

print(f'Feature matrix: train {Xs_train.shape}, OOS {Xs_oos.shape}')
print(f'Class balance (train): {y_train.sum()} wins, {(1-y_train).sum()} losses  ({y_train.mean():.1%} win rate)')

# ── Fit L1 Logistic Regression with cross-validated C ─────────────────────────
print('\nFitting L1 LogisticRegressionCV (5-fold, scoring=roc_auc) ...')
cv_model = LogisticRegressionCV(
    Cs           = CV_C_GRID,
    penalty      = 'l1',
    solver       = 'liblinear',
    class_weight = CLASS_WEIGHT,
    cv           = 5,
    scoring      = 'roc_auc',
    max_iter     = 1000,
    random_state = 42,
)
cv_model.fit(Xs_train, y_train)

best_C = float(cv_model.C_[0])
print(f'Best C selected by CV: {best_C}  (from {CV_C_GRID})')
print(f'(Smaller C = more regularization = fewer non-zero features)')

# ── Coefficient table ──────────────────────────────────────────────────────────
coef = cv_model.coef_[0]
coef_df = pd.DataFrame({
    'Feature'   : FEATURES,
    'Coef'      : coef.round(4),
    'Kept (L1)' : coef != 0,
    'Direction' : ['WIN' if c > 0 else ('LOSE' if c < 0 else 'ZEROED') for c in coef],
}).sort_values('Coef', ascending=False)

print('\nL1 Coefficients (scaled features — non-zero = selected by L1):')
print('  Coef > 0 → signal increases P(win) | Coef < 0 → decreases P(win)')
print()
print(coef_df.to_string(index=False))

# ── Training metrics ───────────────────────────────────────────────────────────
train_probs = cv_model.predict_proba(Xs_train)[:, 1]
train_auc   = roc_auc_score(y_train, train_probs)
n_kept      = int((coef != 0).sum())

print(f'\nTraining AUC        : {train_auc:.3f}  (0.5 = random, 1.0 = perfect)')
print(f'Features retained   : {n_kept}/{len(FEATURES)} (L1 zeroed {len(FEATURES)-n_kept})')
print(f'Prob range (train)  : [{train_probs.min():.3f}, {train_probs.max():.3f}]')

Feature matrix: train (254, 13), OOS (128, 13)
Class balance (train): 62 wins, 192 losses  (24.4% win rate)

Fitting L1 LogisticRegressionCV (5-fold, scoring=roc_auc) ...
Best C selected by CV: 0.5  (from [0.01, 0.05, 0.1, 0.2, 0.5, 1.0])
(Smaller C = more regularization = fewer non-zero features)

L1 Coefficients (scaled features — non-zero = selected by L1):
  Coef > 0 → signal increases P(win) | Coef < 0 → decreases P(win)

               Feature    Coef  Kept (L1) Direction
                us_ret  0.3426       True       WIN
               gap_pct  0.2784       True       WIN
       VIX_INDIA_level  0.1793       True       WIN
        gap_normalized  0.0764       True       WIN
                   dte  0.0291       True       WIN
        prev_india_ret  0.0000      False    ZEROED
        log_entry_prem  0.0000      False    ZEROED
         nifty_20d_ret -0.0495       True      LOSE
            VIX_US_ret -0.0661       True      LOSE
nifty_20d_realized_vol -0.0996       True      LO

In [4]:
# ── Probability distribution check ────────────────────────────────────────────
train_df['prob_win'] = train_probs

print('P(win) distribution on training data:')
print(f'  Min   : {train_probs.min():.3f}')
print(f'  Median: {np.median(train_probs):.3f}')
print(f'  Mean  : {train_probs.mean():.3f}  (≈ base win rate by construction)')
print(f'  Max   : {train_probs.max():.3f}')
print()

# ── Threshold sweep ────────────────────────────────────────────────────────────
thresholds = np.arange(0.24, 0.60, 0.02).round(2)
rows = []
for thr in thresholds:
    sub = train_df[train_df['prob_win'] >= thr]
    n   = len(sub)
    if n == 0:
        continue
    wins     = int(sub['win'].sum())
    win_rate = wins / n
    rows.append({'Threshold': thr, 'Trades': n, 'Wins': wins,
                 'Win%': round(win_rate * 100, 1),
                 'Edge_pp': round((win_rate - base_win_rate) * 100, 1)})

thr_df = pd.DataFrame(rows)
print('Threshold sweep on TRAINING data (in-sample — select cutoff here, not to judge OOS):')
print(thr_df.to_string(index=False))
print()

# ── Auto-select threshold ──────────────────────────────────────────────────────
target_wr  = base_win_rate + EDGE_TARGET_PP / 100
candidates = thr_df[
    (thr_df['Win%'] >= target_wr * 100) &
    (thr_df['Trades'] >= MIN_THRESH_TRADES)
]

if len(candidates) > 0:
    auto_thr = float(candidates['Threshold'].iloc[-1])
    print(f'Auto-threshold: {auto_thr}  (highest threshold with Win% >= {target_wr:.1%} and >= {MIN_THRESH_TRADES} trades)')
else:
    fallback = thr_df[(thr_df['Edge_pp'] > 0) & (thr_df['Trades'] >= 10)]
    auto_thr = float(fallback['Threshold'].iloc[-1]) if len(fallback) > 0 else 0.28
    print(f'WARNING: No threshold achieves +{EDGE_TARGET_PP}pp with {MIN_THRESH_TRADES}+ trades.')
    print(f'Fallback threshold: {auto_thr}')

selected_thr = PROB_THRESHOLD if PROB_THRESHOLD is not None else auto_thr
print(f'Selected threshold: {selected_thr}')

ref = thr_df[thr_df['Threshold'] == selected_thr]
if not ref.empty:
    r = ref.iloc[0]
    print(f'In-sample at this threshold: {int(r["Trades"])} trades, {r["Win%"]:.1f}% win, edge={r["Edge_pp"]:+.1f}pp')

P(win) distribution on training data:
  Min   : 0.028
  Median: 0.441
  Mean  : 0.435  (≈ base win rate by construction)
  Max   : 0.851

Threshold sweep on TRAINING data (in-sample — select cutoff here, not to judge OOS):
 Threshold  Trades  Wins  Win%  Edge_pp
      0.24     233    62  26.6      2.2
      0.26     225    61  27.1      2.7
      0.28     219    61  27.9      3.4
      0.30     211    60  28.4      4.0
      0.32     208    60  28.8      4.4
      0.34     192    58  30.2      5.8
      0.36     184    57  31.0      6.6
      0.38     173    53  30.6      6.2
      0.40     153    49  32.0      7.6
      0.42     143    48  33.6      9.2
      0.44     129    43  33.3      8.9
      0.46     114    39  34.2      9.8
      0.48      94    34  36.2     11.8
      0.50      79    26  32.9      8.5
      0.52      64    20  31.2      6.8
      0.54      54    16  29.6      5.2
      0.56      42    14  33.3      8.9
      0.58      35    12  34.3      9.9

Auto-threshold: 

In [5]:
# ── Compounding backtest engine ────────────────────────────────────────────────
def round_trip_charges(entry_prem: float, exit_prem: float, lots: int) -> float:
    buy_val  = entry_prem * lots * LOT_SIZE
    sell_val = exit_prem  * lots * LOT_SIZE
    brok  = 20.0 * 2
    stamp = 0.00003  * buy_val
    stt   = 0.000625 * sell_val
    exch  = 0.00053  * (buy_val + sell_val)
    sebi  = 0.000001 * (buy_val + sell_val)
    gst   = 0.18     * (brok + exch + sebi)
    return round(brok + stamp + stt + exch + sebi + gst, 2)


def run_backtest(period_label: str, subset: pd.DataFrame, probs: np.ndarray):
    """Compounding capital tracker using cached sim data + model probabilities."""
    skipped_prob = 0
    ledger_rows  = []
    capital, peak = STARTING_CAPITAL, STARTING_CAPITAL

    for i, (_, row) in enumerate(subset.iterrows()):
        prob = float(probs[i])
        if prob < selected_thr:
            skipped_prob += 1
            continue

        ep  = float(row['entry_prem'])
        xp  = float(row['exit_prem'])
        dte = int(row['dte'])

        cost_lot = ep * LOT_SIZE
        if cost_lot <= 0:
            continue
        lots = max(BASE_LOTS, int(capital // cost_lot))
        lots = min(lots, MAX_LOTS)
        if dte == 0:
            lots = min(lots, DTE0_MAX_LOTS)

        charges   = round_trip_charges(ep, xp, lots)
        trade_pnl = (xp - ep) * LOT_SIZE * lots - charges
        capital  += trade_pnl
        peak      = max(peak, capital)

        ledger_rows.append({
            'Trade#'   : len(ledger_rows) + 1,
            'Date'     : row['date'],
            'P(win)'   : round(prob, 3),
            'DTE'      : dte,
            'Lots'     : lots,
            'Entry'    : ep,
            'Exit'     : xp,
            'PnL(pts)' : round(xp - ep, 2),
            'Charges'  : charges,
            'Trade PnL': round(trade_pnl, 2),
            'Capital'  : round(capital, 2),
            'DD%'      : round((peak - capital) / peak * 100, 2),
            'Reason'   : row['exit_reason'],
        })

    if not ledger_rows:
        print(f'{period_label}: 0 trades above threshold {selected_thr}.')
        return pd.DataFrame()

    ledger   = pd.DataFrame(ledger_rows)
    wins     = (ledger['Trade PnL'] > 0).sum()
    total    = len(ledger)
    roi      = (capital - STARTING_CAPITAL) / STARTING_CAPITAL * 100
    max_dd   = ledger['DD%'].max()
    avg_win  = ledger.loc[ledger['Trade PnL'] > 0,  'Trade PnL'].mean() if wins > 0     else 0.0
    avg_loss = ledger.loc[ledger['Trade PnL'] <= 0, 'Trade PnL'].mean() if wins < total else 0.0

    print(f'\n{"="*60}')
    print(f'  {period_label}')
    print(f'{"="*60}')
    print(f'  Below threshold (skipped) : {skipped_prob}')
    print(f'  Above threshold (traded)  : {total}')
    print(f'  Win rate                  : {wins/total*100:.1f}%  (base: {base_win_rate:.1%}, breakeven: {BREAKEVEN:.1%})')
    print(f'  ROI                       : {roi:+.1f}%')
    print(f'  Max drawdown              : {max_dd:.1f}%')
    print(f'  Avg win / avg loss        : Rs {avg_win:,.0f} / Rs {avg_loss:,.0f}')
    print(f'{"="*60}')
    print(ledger['Reason'].value_counts().to_string())
    print()
    print(ledger[['Trade#','Date','P(win)','DTE','Lots','Entry','Exit',
                  'PnL(pts)','Trade PnL','Capital','DD%','Reason']].to_string(index=False))
    return ledger


# ── Run OOS ───────────────────────────────────────────────────────────────────
oos_probs    = cv_model.predict_proba(Xs_oos)[:, 1]
ledger_oos   = run_backtest(
    f'OOS  {OOS_START} → {oos_df["date"].max()}  (out-of-sample)', oos_df, oos_probs)

# ── Run in-sample reference ───────────────────────────────────────────────────
ledger_train = run_backtest(
    f'TRAIN  2024 – {TRAIN_END}  (in-sample reference)', train_df, train_probs)


  OOS  2025-07-01 → 2026-03-24  (out-of-sample)
  Below threshold (skipped) : 118
  Above threshold (traded)  : 10
  Win rate                  : 40.0%  (base: 24.4%, breakeven: 27.3%)
  ROI                       : +8.4%
  Max drawdown              : 21.0%
  Avg win / avg loss        : Rs 38,453 / Rs -22,835
Reason
Target Hit    4
Stop Loss     4
11:15 exit    2

 Trade#       Date  P(win)  DTE  Lots  Entry   Exit  PnL(pts)  Trade PnL   Capital   DD%     Reason
      1 2025-07-02   0.611    1    25  56.40  78.96     22.56   41998.07 241998.07  0.00 Target Hit
      2 2025-09-04   0.668    5    25  68.85  58.52    -10.33  -19638.04 222360.03  8.11  Stop Loss
      3 2025-11-19   0.638    6    25 109.25  92.86    -16.39  -31130.86 191229.17 20.98  Stop Loss
      4 2025-12-17   0.606    6    25  78.70 110.18     31.48   58622.35 249851.52  0.00 Target Hit
      5 2026-01-22   0.708    5    25 109.15  92.78    -16.37  -31093.05 218758.47 12.44  Stop Loss
      6 2026-02-03   0.987    0   

In [6]:
# ── Summary comparison table ───────────────────────────────────────────────────
def _summarize(ledger, label, capital_end=None):
    if ledger.empty:
        return {'Period': label, 'Trades': 0, 'Win%': 'N/A', 'ROI': 'N/A', 'MaxDD': 'N/A'}
    wins  = (ledger['Trade PnL'] > 0).sum()
    total = len(ledger)
    end_cap = ledger['Capital'].iloc[-1] if capital_end is None else capital_end
    roi   = (end_cap - STARTING_CAPITAL) / STARTING_CAPITAL * 100
    maxdd = ledger['DD%'].max()
    return {
        'Period': label, 'Trades': total,
        'Win%'  : f'{wins/total*100:.1f}%',
        'ROI'   : f'{roi:+.1f}%',
        'MaxDD' : f'{maxdd:.1f}%',
    }

summary = pd.DataFrame([
    _summarize(ledger_train, f'v7 TRAIN 2024–Jun 2025    (in-sample)'),
    _summarize(ledger_oos,   f'v7 OOS   Jul 2025–{oos_df["date"].max()}  (out-of-sample)'),
])

print()
print('=' * 70)
print('  v7 — L1 LOGISTIC REGRESSION, CONTINUOUS FEATURES')
print('=' * 70)
print(summary.to_string(index=False))
print('=' * 70)
print('  v6 reference (binary logistic, threshold=0.48, same split):')
print('  TRAIN 2024–Jun 2025  : 15 trades | 60.0% win | +117.1% ROI | 27.5% DD')
print('  OOS Jul 2025–Mar 2026:  5 trades | 40.0% win |   -2.2% ROI | 14.9% DD')
print()
print('  v43 reference (in-sample, 2024–Apr 2026):')
print('  All years            : 120 trades | 33.3% win | +147.7% ROI | 34.2% DD')
print('=' * 70)
print()
print(f'  Model          : L1 LogisticRegressionCV (liblinear)')
print(f'  Best C (CV)    : {best_C}')
print(f'  Class weight   : {CLASS_WEIGHT}')
print(f'  Features kept  : {n_kept}/{len(FEATURES)} (L1 zeroed {len(FEATURES)-n_kept})')
print(f'  Training AUC   : {train_auc:.3f}')
print(f'  Prob threshold : {selected_thr}')
print(f'  Base win rate  : {base_win_rate:.1%}  |  Breakeven: {BREAKEVEN:.1%}')
print(f'  Train days     : {len(train_df)}  |  OOS days: {len(oos_df)}')


  v7 — L1 LOGISTIC REGRESSION, CONTINUOUS FEATURES
                                       Period  Trades  Win%   ROI MaxDD
        v7 TRAIN 2024–Jun 2025    (in-sample)      35 37.1% -9.1% 56.6%
v7 OOS   Jul 2025–2026-03-24  (out-of-sample)      10 40.0% +8.4% 21.0%
  v6 reference (binary logistic, threshold=0.48, same split):
  TRAIN 2024–Jun 2025  : 15 trades | 60.0% win | +117.1% ROI | 27.5% DD
  OOS Jul 2025–Mar 2026:  5 trades | 40.0% win |   -2.2% ROI | 14.9% DD

  v43 reference (in-sample, 2024–Apr 2026):
  All years            : 120 trades | 33.3% win | +147.7% ROI | 34.2% DD

  Model          : L1 LogisticRegressionCV (liblinear)
  Best C (CV)    : 0.5
  Class weight   : {0: 1.0, 1: 2.5}
  Features kept  : 11/13 (L1 zeroed 2)
  Training AUC   : 0.667
  Prob threshold : 0.58
  Base win rate  : 24.4%  |  Breakeven: 27.3%
  Train days     : 254  |  OOS days: 128


In [7]:
# ── Export model bundle for cron/v7 ──────────────────────────────────────────
# Run this cell AFTER all other cells to save the trained model.
# Copy the output file v7_model.pkl → cron/v7/ before deploying to EC2.

import joblib

MODEL_SAVE_PATH = Path.cwd() / 'v7_model.pkl'

model_bundle = {
    'model':         cv_model,        # fitted LogisticRegressionCV
    'scaler':        scaler,          # fitted StandardScaler (must apply before predict)
    'threshold':     selected_thr,    # P(win) cutoff for trade entry
    'features':      FEATURES,        # feature names in exact order (must match entry.py)
    'train_end':     str(TRAIN_END),
    'base_win_rate': round(base_win_rate, 4),
    'best_C':        best_C,
    'train_auc':     round(train_auc, 3),
}

joblib.dump(model_bundle, MODEL_SAVE_PATH)

print(f'Model bundle saved → {MODEL_SAVE_PATH}')
print(f'  Model       : {type(cv_model).__name__}  (C={best_C}, penalty=l1)')
print(f'  Threshold   : {selected_thr}')
print(f'  Train AUC   : {train_auc:.3f}')
print(f'  Train end   : {TRAIN_END}')
print(f'  Features    : {FEATURES}')
print()
print('  Next step: copy v7_model.pkl → gap_trading/cron/v7/v7_model.pkl')

Model bundle saved → c:\Users\sayan\OneDrive\Desktop\Projects\03_Market_Research\market-research\gap_trading\v7\v7_model.pkl
  Model       : LogisticRegressionCV  (C=0.5, penalty=l1)
  Threshold   : 0.58
  Train AUC   : 0.667
  Train end   : 2025-06-30
  Features    : ['gap_pct', 'prev_india_ret', 'us_ret', 'europe_ret', 'asia_ret', 'VIX_US_ret', 'VIX_US_level', 'VIX_INDIA_level', 'log_entry_prem', 'dte', 'nifty_20d_ret', 'nifty_20d_realized_vol', 'gap_normalized']

  Next step: copy v7_model.pkl → gap_trading/cron/v7/v7_model.pkl
